# 01_entrenamiento - Opción 2 Python + Streamlit
Notebook resumido para entrenar RNA, comparar con GBT y generar métricas.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report


In [ ]:
df = pd.read_csv('diabetes_data_es_GOLD.csv')
print(df.shape)
print(df['TieneDiabetes'].value_counts())
le = LabelEncoder()
df['target'] = le.fit_transform(df['TieneDiabetes'])
X = df.drop(columns=['TieneDiabetes','target']).values
y = df['target'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:
corr = df.drop(columns=['TieneDiabetes']).corr()['target'].sort_values(ascending=False)
print(corr)
plt.figure(figsize=(8,6))
corr.drop('target').plot(kind='barh')
plt.title('Correlación de cada feature con TieneDiabetes')
plt.tight_layout(); plt.show()


In [ ]:
rna = MLPClassifier(hidden_layer_sizes=(32,16), activation='relu', solver='adam', max_iter=700, random_state=42, early_stopping=True, validation_fraction=0.15)
rna.fit(X_train, y_train)
y_prob_rna = rna.predict_proba(X_test)[:,1]
y_pred_rna = (y_prob_rna >= 0.5).astype(int)


In [ ]:
gbt = GradientBoostingClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
gbt.fit(X_train, y_train)
y_pred_gbt = gbt.predict(X_test)
y_prob_gbt = gbt.predict_proba(X_test)[:,1]
res = pd.DataFrame({
    'Modelo':['RNA MLP [32,16]','Gradient Boosted Trees'],
    'Accuracy':[accuracy_score(y_test,y_pred_rna), accuracy_score(y_test,y_pred_gbt)],
    'F1':[f1_score(y_test,y_pred_rna), f1_score(y_test,y_pred_gbt)],
    'AUC':[roc_auc_score(y_test,y_prob_rna), roc_auc_score(y_test,y_prob_gbt)]
})
print(res)


In [ ]:
fig, axes = plt.subplots(1,2,figsize=(10,4))
for ax, pred, title in [(axes[0], y_pred_rna, 'RNA'), (axes[1], y_pred_gbt, 'GBT')]:
    sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt='d', ax=ax, xticklabels=['Neg','Pos'], yticklabels=['Neg','Pos'])
    ax.set_title('Matriz de confusión - ' + title)
plt.tight_layout(); plt.show()


In [ ]:
importances = pd.Series(gbt.feature_importances_, index=df.drop(columns=['TieneDiabetes','target']).columns).sort_values(ascending=False)
print(importances)
importances.plot(kind='barh', figsize=(8,6), title='Feature importances - GBT')
plt.tight_layout(); plt.show()
